In [ ]:
import logging
import itertools
from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.qubit_array import QubitArray
from library.steane_code.compact_patch import SteaneCodePatch
from utils.circuit_expectations import count_cnots
from utils.circuit_flows import check_state_preparation

logging.basicConfig(level=logging.ERROR)

In [ ]:
qubits = QubitArray(dimensions=(5, 5))
circuitry = Circuitry(qubits)
steane = SteaneCodePatch(qubits, anchor=(1, 1))

circuitry.annotate_polygons(steane.get_polygons(compact=False))
circuitry.append_tick()
steane.append_preparation(circuitry)
steane.append_superdense_cycle(circuitry, prefix="STN:SDC0")
steane.append_cultivation(circuitry)
for rnd in range(3):
    steane.append_teleportation_round(circuitry, round=rnd, prefix=f"STN:TPT{rnd}")
steane.append_destruction(circuitry)

steane.annotate_detectors(circuitry, sdc_rounds=1, tpt_rounds=3)

display(Markdown(f"[Open in Crumble]({circuitry.to_crumble_url()})"))

In [ ]:
circuitry.detectors_report()

In [ ]:
for stabilizer, (plaquette, support) in itertools.product(
    ["X", "Z"], steane.stabilizers().items()
):
    check_state_preparation(circuitry.as_stim(), stabilizer, support, label=plaquette)
check_state_preparation(circuitry.as_stim(), "Y", support=steane.support(), label="O")
count_cnots(circuitry.as_stim)

In [ ]:
for record in qubits.measurements_index:
    print(f"Record : {record}")